# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the FAIR² dataset
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print basic dataset metadata
print(f"Dataset Title: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their `@id` values, fields, and columns.

**Note:** To ensure accurate referencing, all entities are accessed using their `@id`.

In [ ]:
# View all available record sets and their details
record_sets = list(dataset.metadata.record_sets)
print(f"Total record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet: {rs['@id']}")
    print(f"  Name: {rs.get('name', '')}")
    print(f"  Description: {rs.get('description', '')}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print(f"  Fields: {fields}")
    print("")

**Example record preview:**

Let's print a sample from each record set using their `@id` (replace with actual `@id` from above).

In [ ]:
# Print a preview of records from each record set by `@id`
for rs in record_sets:
    print(f"Preview records for RecordSet: {rs['@id']}")
    try:
        records = dataset.records(record_set=rs['@id'])
        for i, rec in enumerate(records):
            print(rec)
            if i >= 2:  # Show only first 3 records
                break
    except Exception as e:
        print(f"Could not load records: {e}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We'll extract all record sets, using their `@id` values. Fields and column names will remain referenced by their `@id` for clarity.

In [ ]:
# Extract records from all record sets into dataframes
dataframes = {}
for rs in record_sets:
    rec_id = rs['@id']
    print(f"Loading records for RecordSet: {rec_id}")
    try:
        df = pd.DataFrame(list(dataset.records(record_set=rec_id)))
        dataframes[rec_id] = df
        print(f"Loaded DataFrame shape: {df.shape}")
        print(f"Columns (@id): {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"Could not create DataFrame for {rec_id}: {e}")
    print("")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping. All operations reference columns by their `@id`.

In [ ]:
# Choose one record set for EDA (first available)
main_rs = record_sets[0]['@id'] if record_sets else None
df = dataframes[main_rs]
print(f"Using RecordSet {main_rs} for EDA.")

# Find likely numeric fields (columns) by looking for numbers in first row
sample_row = df.iloc[0] if not df.empty else {}
numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not numeric_candidates:
    # Try to coerce numeric columns if needed
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

print(f"Numeric columns (@id): {numeric_candidates}")

# For demonstration, use the first found numeric field
if numeric_candidates:
    numeric_field = numeric_candidates[0]
    threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records in '{main_rs}' with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field}:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by a non-numeric field (the first found)
    group_candidates = [col for col in df.columns if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col])]
    if group_candidates:
        group_field = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped by '{group_field}' (mean of {numeric_field}):")
        print(grouped_df.head())
    else:
        print("No non-numeric columns available for grouping.")
else:
    print("No numeric columns found for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field or the relationship with a grouping field.

You may extend this section by choosing fields of interest by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'numeric_field' in locals() and not df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {main_rs}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If group_field is available, plot grouped mean
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10,4))
        df_grouped = df.groupby(group_field)[numeric_field].mean().sort_values()
        df_grouped.plot(kind='bar')
        plt.title(f"Mean {numeric_field} per {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset using the `mlcroissant` library, identified key record sets and fields, carried out basic data filtering and normalization, and visualized important trends.

All data elements (record sets, fields, columns) were referenced by their `@id` according to the Croissant specification. For advanced use, refer to the `mlcroissant` [documentation](https://mlcommons.org/croissant/) to handle custom extraction or integration tasks.